# Semantic Bridge Analysis

## A TACC Computational Cookbook

**Developed by:** Texas Advanced Computing Center (TACC)  
**Institution:** The University of Texas at Austin  
**Contact:** [DSO@tacc.utexas.edu](mailto:DSO@tacc.utexas.edu)

---

### What This Cookbook Does

This tutorial demonstrates how to move from qualitative stakeholder narratives to quantitative scientific framing using natural language processing and lightweight machine learning.

By the end of the workflow, you will be able to:
1. discover themes in stakeholder documents
2. extract decision components such as goals, objectives, variables, and constraints
3. map those themes to scientific domains
4. connect plain-language concepts to scientific variables
5. generate outputs that support downstream decision analysis

Typical use cases include environmental planning, climate adaptation, infrastructure decision support, participatory modeling, and interdisciplinary problem framing.

### Prerequisites

- A TACC account and allocation, or another environment prepared from this repository
- Basic familiarity with Jupyter notebooks and Python
- A set of documents that describe a planning or decision problem

Supported inputs include `.txt`, `.json`, `.docx`, `.pdf`, and common image formats for OCR-based extraction.

### Quick Start

1. Run the setup cells to verify the managed environment.
2. Place source documents in `data/subsidence_groundwater_corpus/raw/` or cleaned text in `data/subsidence_groundwater_corpus/cleaned/`.
3. Work through the topic discovery, science mapping, and decision-component extraction steps.
4. Review the generated figures, tables, and summary report in `outputs/`.

### Expected Outputs

- `*_topic_mappings.csv`: topics linked to scientific domains
- `*_components.csv`: extracted decision components
- `*_svo_mappings.csv`: semantic links to scientific variables
- `*_network.html`: interactive science-backbone visualization
- `*_report.md`: summary report for sharing and review

### Customization

- Update `science_backbone` to reflect your domain structure.
- Expand `svo_vocabulary` with variables relevant to your project.
- Adjust `n_topics` to match the size and diversity of your document set.
- Refine the decision-component extraction patterns for your context.

### Learn More

- [TACC Documentation](https://docs.tacc.utexas.edu)
- [TACC Training](https://learn.tacc.utexas.edu)
- [Science Gateways Community Institute](https://sciencegateways.org/)



In [ ]:
import os
from pathlib import Path

import pandas as pd
import spacy

import semantic_bridge_notebook_utils as nbutils
import semantic_bridge_pipeline as sbp

nlp = spacy.load('en_core_web_sm')
print('Libraries loaded successfully.')



## Step 2: Prepare Input Data

In this step, you will assemble the document collection, or corpus, that describes the problem you want to analyze.

Typical examples include:
- Interview transcripts
- Meeting notes
- Stakeholder reports
- Grey literature reports
- Community narratives

**Supported formats:**
- `.txt` - Plain text files
- `.json` - JSON with text content (field: `text`, `content`, `body`, or `description`)
- `.docx` - Microsoft Word documents
- `.pdf` - PDFs with extractable text
- `.png` / `.jpg` / `.jpeg` / `.tif` / `.tiff` - Images for OCR extraction

**Corpus layout:**
- Put source files in `data/subsidence_groundwater_corpus/raw/`
- Put cleaned text files in `data/subsidence_groundwater_corpus/cleaned/` if you already have them
- The loader will prefer `cleaned/` when both versions exist and otherwise fall back to `raw/`

For this tutorial, you can either use your own files or run the next cell to generate sample documents in `raw/`.



### Locate the Working Directories

This cell finds the tutorial folder on disk and identifies the corpus directories the rest of the notebook will use. It also decides whether to work from the cleaned text folder or the raw source folder.


In [ ]:
tutorial_dir = nbutils.resolve_tutorial_dir()
corpus_dir = sbp.ensure_data_directory(tutorial_dir / 'data' / 'subsidence_groundwater_corpus')
raw_dir = sbp.ensure_data_directory(corpus_dir / 'raw')
cleaned_dir = sbp.ensure_data_directory(corpus_dir / 'cleaned')

data_dir = cleaned_dir if any(cleaned_dir.iterdir()) else raw_dir
nbutils.print_runtime_paths(tutorial_dir, corpus_dir, raw_dir, cleaned_dir, data_dir)



tutorial_dir = resolve_tutorial_dir()
corpus_dir = ensure_data_directory(tutorial_dir / 'data' / 'subsidence_groundwater_corpus')
raw_dir = ensure_data_directory(corpus_dir / 'raw')
cleaned_dir = ensure_data_directory(corpus_dir / 'cleaned')

data_dir = cleaned_dir if any(cleaned_dir.iterdir()) else raw_dir
print_runtime_paths(tutorial_dir, corpus_dir, raw_dir, cleaned_dir, data_dir)



In [ ]:
transcripts = sbp.load_documents(corpus_dir)
nbutils.print_document_list(transcripts)




### Inspect the Input Text

Before modeling anything, it helps to look at a short preview from each document. This gives you a quick sense of writing style, data quality, and whether the right files were loaded.


In [ ]:
nbutils.print_document_previews(sbp.preview_documents(transcripts))




### Check Corpus Size

This cell reports basic corpus statistics such as document length, word counts, and sentence counts. Use it to confirm that the documents are substantial enough for topic discovery.


In [ ]:
stats_df = sbp.build_stats_table(transcripts)

print('✓ Transcript Statistics:\n')
print(stats_df.to_string(index=False))



## Step 2a: Configure Filler Words

Use this cell to remove transcript fillers or domain-specific words that would otherwise dominate the topic model.

Examples include hesitation words such as `um` and `uh`, or repeated interview artifacts such as speaker labels.


### Define Filler Words to Exclude

Use this cell to remove conversational fillers or repeated artifacts that are not analytically useful. These words will be filtered before topic modeling so they do not dominate the learned themes.


In [ ]:
CUSTOM_STOPWORDS = {
    'um',
    'uh',
    'like',
    'you',
    'know',
}

print('Custom filler-word list:')
print(sorted(CUSTOM_STOPWORDS))



## Step 3: Discover Topics

**What is topic modeling?**

Topic modeling automatically identifies themes in a document collection by grouping together words that often appear in the same context.

For example, if words like `flooding`, `water`, and `drainage` frequently occur together, the model may identify a topic related to stormwater or hydrology.

**How it works:**
1. Break text into words
2. Find patterns of co-occurring words
3. Group related words into topics
4. Assign topics to documents

**Parameters used in this step:**
- `n_topics` - The number of topics the model will try to discover across the corpus. Increase it for more thematic detail; decrease it for broader themes.
- `max_vocabulary` - The maximum number of unique words or phrases the model is allowed to consider when building topics. This limits the feature space to the most informative terms after preprocessing.
- `top_words_display` - The number of top-weighted keywords to show when summarizing each topic in the notebook output.

**How to think about `max_vocabulary`:**
- A larger value allows the model to consider more distinct terms.
- A smaller value forces the model to focus on a tighter set of common or high-signal terms.
- If this value is too small, important concepts may be excluded.
- If it is too large, the model may include more noisy or low-value terms.

For small tutorial corpora, values around `100` are usually fine. Larger, more diverse corpora often benefit from higher values.



### Set Topic Parameters

Adjust these values to control how the topic model behaves. This is the main place to tune the level of thematic detail you want from the corpus.


In [ ]:
n_topics = 5
max_vocabulary = 200
top_words_display = 12
nbutils.print_topic_parameters(n_topics, max_vocabulary, top_words_display)




### Preprocess the Text

This cell cleans and normalizes the documents so the topic model can focus on meaningful terms rather than punctuation, capitalization, or filler language.


In [ ]:
processed_docs, doc_names = sbp.preprocess_documents(
    transcripts,
    custom_stopwords=CUSTOM_STOPWORDS,
)

print('✓ Text preprocessing complete')
print(f'\nExample: {processed_docs[0][:150]}...')



### Run Topic Discovery

This is the main topic-modeling step. The model identifies groups of terms that tend to appear together and uses them to summarize the major themes in the corpus.


In [ ]:
topic_results = sbp.discover_topics(
    processed_docs,
    n_topics=n_topics,
    max_vocabulary=max_vocabulary,
    topic_keyword_count=top_words_display,
    custom_stopwords=CUSTOM_STOPWORDS,
)

doc_topic_dist = topic_results['doc_topic_dist']
topics_info = topic_results['topics_info']
feature_names = topic_results['feature_names']
nbutils.print_topic_discovery_summary(topics_info, feature_names, top_words_display)




## Step 3a: Optional LLM Topic Relabeling

The topic model discovers clusters of terms, but those raw keyword labels are often awkward.

This step is optional. If enabled, the notebook reruns topic discovery to capture the baseline keyword labels, then asks an OpenAI-compatible model for a cleaner label and a one-sentence description so you can compare the two versions directly.


### Configure Optional Topic Relabeling

If you have access to an OpenAI-compatible model endpoint, you can use it to rewrite raw keyword topics into clearer labels and descriptions. Leave this disabled if you want to stay fully local.


In [ ]:
ENABLE_LLM_TOPIC_LABELS = True
TOPIC_LABELER_MODEL = 'Meta-Llama-3.3-70B-Instruct'
TOPIC_LABELER_BASE_URL = os.getenv('OPENAI_BASE_URL', 'https://ai.tejas.tacc.utexas.edu')
TOPIC_LABELER_API_KEY = os.getenv('OPENAI_API_KEY', '')

print(f'LLM relabeling enabled: {ENABLE_LLM_TOPIC_LABELS}')
print(f'Model: {TOPIC_LABELER_MODEL}')
print(f'Base URL configured: {bool(TOPIC_LABELER_BASE_URL)}')
print(f'API key configured: {bool(TOPIC_LABELER_API_KEY)}')




### Compare Raw Topics to Human-Readable Topic Labels

This optional cell shows the difference between the model's original keyword-based topic labels and the rewritten labels generated by the LLM. It is useful for teaching how machine-generated clusters can be translated into more readable summaries.


In [ ]:
if ENABLE_LLM_TOPIC_LABELS:
    if not TOPIC_LABELER_MODEL:
        raise ValueError('Set TOPIC_LABELER_MODEL before enabling LLM relabeling.')
    if not TOPIC_LABELER_API_KEY:
        raise ValueError('Set OPENAI_API_KEY before enabling LLM relabeling.')

    baseline_topic_results = sbp.discover_topics(
        processed_docs,
        n_topics=n_topics,
        max_vocabulary=max_vocabulary,
        topic_keyword_count=top_words_display,
        custom_stopwords=CUSTOM_STOPWORDS,
    )
    baseline_topics_info = baseline_topic_results['topics_info']

    relabeled_topics_info = sbp.relabel_topics_with_llm(
        topics_info=baseline_topics_info,
        doc_topic_dist=doc_topic_dist,
        doc_names=doc_names,
        documents=transcripts,
        model=TOPIC_LABELER_MODEL,
        api_key=TOPIC_LABELER_API_KEY,
        base_url=TOPIC_LABELER_BASE_URL or None,
    )

    nbutils.print_topic_label_comparison(baseline_topics_info, relabeled_topics_info, top_words_display)
    topics_info = relabeled_topics_info
else:
    print('Skipping LLM topic relabeling.')




## Step 3b: Recommend MINT Queries for Each Topic

This optional step uses the MINT model catalog to suggest which model domains, tags, and candidate models are worth querying next for each topic.


### Suggest MINT Queries for Each Topic

This optional cell looks at each topic and recommends which MINT model domains, tags, and candidate models may be relevant. Use it as a starting point for model discovery rather than as a final answer.


In [ ]:
USE_MINT_TOPIC_RECOMMENDATIONS = True
MINT_TOPIC_API_BASE_URL = 'https://api.models.mint.tacc.utexas.edu/v1.8.0'
MINT_TOPIC_USERNAME = 'mint@isi.edu'
MINT_TOPIC_PER_PAGE = 100
MINT_TOPIC_MAX_PAGES = 3
MINT_TOPIC_MODELS_PER_TOPIC = 2
MINT_TOPIC_TAGS_PER_TOPIC = 5

topic_query_recommendations = []

if USE_MINT_TOPIC_RECOMMENDATIONS:
    try:
        mint_topic_candidates = sbp.fetch_mint_model_candidates(
            base_url=MINT_TOPIC_API_BASE_URL,
            username=MINT_TOPIC_USERNAME,
            per_page=MINT_TOPIC_PER_PAGE,
            max_pages=MINT_TOPIC_MAX_PAGES,
        )
        topic_query_recommendations = sbp.recommend_mint_queries_for_topics(
            topics_info,
            mint_topic_candidates,
            recommendations_per_topic=MINT_TOPIC_MODELS_PER_TOPIC,
            tags_per_topic=MINT_TOPIC_TAGS_PER_TOPIC,
        )
        nbutils.print_topic_query_recommendations(topic_query_recommendations)
    except Exception as exc:
        print(f'Could not retrieve MINT topic recommendations: {exc}')
else:
    print('Skipping MINT topic recommendations.')




### Visualize Topic Emphasis by Document

This chart shows how strongly each topic appears in each document. It helps you see whether a theme is widespread across the corpus or concentrated in just a few files.


In [ ]:
print('Creating topic distribution chart...\n')

topic_df, fig = sbp.plot_topic_distribution(
    doc_topic_dist,
    doc_names,
    topics_info,
    n_topics=n_topics,
)
fig.show()

print('✓ Visualization complete!')
print('\nTip: change n_topics above and rerun Step 3 to compare topic structures.')



### Review the Topic Summary Table

This table gives a compact summary of the discovered topics, including their labels, descriptions, keywords, and average coverage in the corpus.


In [ ]:
summary_df = sbp.build_topic_summary(topics_info, doc_topic_dist, top_words_display)
summary_df




## Step 4: Map to the Science Backbone

This step connects the discovered topics to a simple scientific domain structure.

**What is the science backbone?**

Think of the science backbone as a lightweight hierarchy of domains and subdomains.

- Major domains: Environmental Science, Social Science, Engineering, and related fields
- Subdisciplines: Hydrology, Economics, Urban Planning, and others
- Specific problem areas: flood modeling, cost-benefit analysis, infrastructure resilience, and more

Mapping topics into this structure helps show which scientific fields are relevant to the stakeholder narratives.

### Configure Output Names

This cell sets the folder and case-study name used when the notebook writes results to disk. Adjust it if you want to save outputs under a different label.


In [ ]:
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
CASE_STUDY_NAME = 'community_analysis'

print(f'✓ Output directory: {OUTPUT_DIR}')
print(f'✓ Case study name: {CASE_STUDY_NAME}')



### Review the Science Backbone

The science backbone is a lightweight domain structure used to organize topics. You can keep the default version for the tutorial or edit it to better reflect your own problem domain.


In [ ]:
science_backbone = sbp.default_science_backbone()
nbutils.print_science_backbone(science_backbone)




### Map Topics to Scientific Domains

This step links each topic to one or two parts of the science backbone. The goal is not to be perfect, but to show how stakeholder narratives can be connected to disciplinary frames.


In [ ]:
topic_mappings = sbp.map_topics_to_domains(topics_info)
mapping_df = pd.DataFrame(topic_mappings)
nbutils.print_topic_mappings(topic_mappings)




### Visualize the Science Backbone Network

This figure places the discovered topics into the science backbone as a simple network. It is useful for explaining how narrative themes connect to broader scientific areas.


In [ ]:
print('Creating network visualization...\n')

graph, network_fig = sbp.create_network_figure(
    science_backbone,
    topic_mappings,
    case_study_name=CASE_STUDY_NAME,
)
network_fig.show()

print('✓ Visualization complete!')



## Step 5: Extract Decision Components

Most decision problems include a common set of elements. In this step, the notebook looks for those elements directly in the text.

We focus on:
- Goals: What are we trying to achieve?
- Objectives: Specific, measurable aims
- Decision Variables: What can we control?
- Constraints: Limits that shape what is feasible
- Indicators: How do we measure success?

### Extract Decision Components

This cell searches the corpus for common elements of a decision problem, such as goals, objectives, constraints, and indicators. It provides a first-pass structure for downstream decision analysis.


In [ ]:
decision_components = sbp.extract_decision_components(transcripts, nlp)
nbutils.print_decision_components(decision_components)




### Compare Decision Component Counts

This bar chart summarizes how many items were found for each decision-component type. It helps you see which parts of the decision framing are well represented in the text.


In [ ]:
component_counts_map = sbp.component_counts(decision_components)
component_fig = sbp.plot_component_distribution(component_counts_map)
component_fig.show()

print('✓ Visualization complete!')



### Build a Reusable Decision-Component Table

This cell turns the extracted decision components into a tabular structure that can be filtered, exported, or reused later in the notebook.


In [ ]:
components_df = sbp.component_table(decision_components)
components_df




## Step 6: Link to Scientific Variables

**What are Scientific Variable Objects (SVOs)?**

An SVO is a standardized description of a measurable quantity.

It helps translate everyday language into scientific variables that can be observed, modeled, or retrieved from data systems.

In this tutorial, the scientific-variable vocabulary now comes from the MINT variable presentations API when available, with a small local fallback vocabulary for offline use.

Each SVO-style entry can include:
- Variable Name: What we measure
- Units: How we measure it
- Data Source: Where the variable definition came from
- Standard Name: Scientific terminology
- Domain: A broad scientific grouping when available

This step creates a translation layer between stakeholder language and measurable scientific concepts.



### Load the Scientific-Variable Vocabulary

This cell retrieves scientific variable definitions from the MINT API when available. If the live service is unavailable, it falls back to a smaller local tutorial vocabulary so the workflow can continue.


In [ ]:
USE_MINT_SVO_VOCABULARY = True
MINT_SVO_API_BASE_URL = 'https://api.models.mint.tacc.utexas.edu/v1.8.0'
MINT_SVO_USERNAME = 'mint@isi.edu'
MINT_SVO_PER_PAGE = 200
MINT_SVO_MAX_PAGES = 3

try:
    if USE_MINT_SVO_VOCABULARY:
        svo_vocabulary = sbp.fetch_mint_svo_vocabulary(
            base_url=MINT_SVO_API_BASE_URL,
            username=MINT_SVO_USERNAME,
            per_page=MINT_SVO_PER_PAGE,
            max_pages=MINT_SVO_MAX_PAGES,
        )
        svo_source = 'MINT variablepresentations API'
    else:
        raise RuntimeError('Configured to use local fallback vocabulary.')
except Exception as exc:
    print(f'Falling back to local demo SVO vocabulary: {exc}')
    svo_vocabulary = sbp.default_svo_vocabulary()
    svo_source = 'Local fallback vocabulary'

nbutils.print_svo_vocabulary_preview(svo_vocabulary, svo_source)




### SVO Matching Options

Use these controls to tighten or loosen how natural-language terms are matched to scientific variables.

- `SVO_MIN_KEYWORD_WORDS` sets the minimum number of words required for an automatic match.
- `SVO_SINGLE_WORD_ALLOWLIST` lets you preserve a small set of strong scientific one-word terms.


### Control How Strictly SVOs Are Matched

Use these settings to decide how cautious the notebook should be when linking natural-language terms to scientific variables. Stricter settings reduce false positives; looser settings can recover more matches.


In [ ]:
SVO_MIN_KEYWORD_WORDS = 2
SVO_SINGLE_WORD_ALLOWLIST = {
    'subsidence',
    'precipitation',
    'salinity',
    'groundwater',
}

print(f'Minimum keyword length: {SVO_MIN_KEYWORD_WORDS} words')
print(f'Allowed single-word scientific terms: {sorted(SVO_SINGLE_WORD_ALLOWLIST)}')



### Link Narrative Terms to Scientific Variables

This cell matches terms from the corpus to scientific variables in the vocabulary. The result is a bridge between everyday language and measurable scientific concepts.


In [ ]:
svo_mappings = sbp.create_svo_mappings(
    transcripts,
    svo_vocabulary,
    min_keyword_words=SVO_MIN_KEYWORD_WORDS,
    allow_single_word_keywords=SVO_SINGLE_WORD_ALLOWLIST,
)
unique_mappings = sbp.deduplicate_svo_mappings(svo_mappings)
svo_df = sbp.svo_table(unique_mappings)
nbutils.print_svo_mapping_summary(unique_mappings)




### MINT Model Recommendations

This optional substep queries the MINT model catalog and recommends up to two models or model configurations for each matched scientific variable.


### Recommend MINT Models for Matched Variables

This optional cell suggests MINT models or model configurations that appear relevant to the scientific variables found in the corpus. Treat these as candidate starting points for further review.


In [ ]:
USE_MINT_MODEL_RECOMMENDATIONS = True
MINT_MODEL_API_BASE_URL = MINT_SVO_API_BASE_URL
MINT_MODEL_USERNAME = MINT_SVO_USERNAME
MINT_MODEL_PER_PAGE = 100
MINT_MODEL_MAX_PAGES = 3
MINT_MODELS_PER_SVO = 2

model_recommendations = []
model_recommendations_df = pd.DataFrame()

if USE_MINT_MODEL_RECOMMENDATIONS and unique_mappings:
    try:
        mint_model_candidates = sbp.fetch_mint_model_candidates(
            base_url=MINT_MODEL_API_BASE_URL,
            username=MINT_MODEL_USERNAME,
            per_page=MINT_MODEL_PER_PAGE,
            max_pages=MINT_MODEL_MAX_PAGES,
        )
        model_recommendations = sbp.recommend_models_for_svo_mappings(
            unique_mappings,
            mint_model_candidates,
            recommendations_per_svo=MINT_MODELS_PER_SVO,
        )
        model_recommendations_df = pd.DataFrame(model_recommendations)
        nbutils.print_svo_model_recommendations(model_recommendations_df)
    except Exception as exc:
        print(f'Could not retrieve MINT model recommendations: {exc}')
else:
    print('Skipping MINT model recommendations.')




### Visualize Scientific Variables by Domain

This chart summarizes which scientific domains are represented by the matched variables. It gives a quick sense of where the narrative is strongest in scientific terms.


In [ ]:
_, sunburst_fig, domain_counts = sbp.plot_svo_sunburst(unique_mappings)
sunburst_fig.show()
nbutils.print_domain_counts(domain_counts)




## Interpreting the Sunburst Figure

The **Scientific Variables by Domain** sunburst chart shows how stakeholder narratives connect to scientific concepts.

### What the figure shows

**Inner ring:** scientific domains such as Hydrology, Climate Science, Social Science, Economics, Engineering, and Oceanography

**Outer ring:** individual scientific variables nested within each domain, such as:
- `water_level` and `groundwater_level` under Hydrology
- `precipitation` under Climate Science  
- `economic_damage` under Economics
- `infrastructure_vulnerability` under Engineering

**Segment size:** proportional to how frequently each domain or variable appears in the mapped text

### What it tells you

1. **Problem scope:** which scientific disciplines are most relevant to the stakeholder concerns

2. **Data needs:** what kinds of measurements, datasets, or monitoring systems may support the decision process

3. **Interdisciplinary structure:** whether the problem is concentrated in one field or spread across several domains

4. **Stakeholder priorities:** which measurable concepts best reflect the language people use to describe the problem

### Why this matters for TACC workflows

This figure helps you identify:
- Which computational models you need (hydrological, economic, infrastructure assessment)
- What datasets to access from TACC resources or external sources
- Which scientific experts should be engaged
- How to structure your decision support system to match stakeholder mental models

In short, it shows which science is needed to answer the questions the community is asking.

## Step 7: Generate Summary Report

In the final step, the notebook writes a short report that summarizes the topics, mappings, decision components, and generated outputs.

### Export the Results

This cell writes the key outputs of the analysis to disk, including tables, figures, and a summary report. It is the main checkpoint for saving the notebook results.


In [ ]:
output_files = sbp.write_outputs_table(
    OUTPUT_DIR,
    CASE_STUDY_NAME,
    topic_mappings,
    components_df,
    svo_df,
    network_fig,
    sunburst_fig,
)

summary = sbp.build_summary_report(
    CASE_STUDY_NAME,
    transcripts,
    n_topics,
    topic_mappings,
    decision_components,
    unique_mappings,
    svo_df,
)
report_path = sbp.write_report(OUTPUT_DIR, CASE_STUDY_NAME, summary)
nbutils.print_export_summary(
    transcripts,
    n_topics,
    topic_mappings,
    decision_components,
    unique_mappings,
    output_files,
    report_path,
)




### Preview the Summary Report

Use this cell to inspect the beginning of the generated report before sharing it or moving it into another workflow.


In [ ]:
nbutils.print_report_preview(summary, report_path)




### Create a Quick Reference Table

This final table condenses the major outputs of the workflow into a single summary view. It can be useful for presentations, handoffs, or rapid review.


In [ ]:
quick_ref = sbp.build_quick_reference(
    transcripts,
    n_topics,
    topic_mappings,
    decision_components,
    svo_vocabulary,
    unique_mappings,
    output_files,
)

quick_ref_path = OUTPUT_DIR / f'{CASE_STUDY_NAME}_summary_table.csv'
quick_ref.to_csv(quick_ref_path, index=False)
quick_ref




## Wrap-Up

You have completed the semantic bridge analysis workflow.

### What you learned
- How to extract topics from text automatically
- How to map narratives to scientific domains
- How to identify decision components
- How to link language to measurable variables

### How to apply this to your own work
1. Replace sample data with your own documents
2. Customize domains in `science_backbone`
3. Expand `svo_vocabulary`
4. Adjust the `n_topics` parameter
5. Re-run and validate with stakeholders

### Output files
All results are saved in the `outputs/` folder:
- CSV files for further analysis
- HTML visualizations for sharing
- Markdown report for documentation

### Next steps
- Review the generated outputs with domain experts and stakeholders
- Refine the science backbone and variable mappings for your domain
- Connect the results to downstream datasets, models, or decision tools

This workflow is designed to preserve stakeholder language while connecting it to scientific and engineering analysis.